# 20.4 Örnek 2: LSTM ile zaman serisi tahmini

*Makine Öğrenmesi Çalışma Rehberi · Bölüm 20: PyTorch ile Uygulamalı Örnekler*

**Nasıl kullanılır?** Hücreleri sırayla çalıştır (Shift + Enter) ya da menüden *Çalışma zamanı → Tümünü çalıştır*'ı seç. GPU gerekmez. Kodu değiştirip yeniden çalıştırmaktan çekinme. Köşeli parantezli numaralar ([76] gibi) PDF'teki kaynakçaya karşılık gelir.

### Veri ve kayan pencere

Bu örnekte veriyi kendimiz üretiyoruz: periyodu 50 adım olan bir sinüs dalgasına, standart sapması 0,1 olan rastgele gürültü ekliyoruz. Bunu, her gün benzer bir döngü izleyen ama ölçüm hataları içeren bir sensör verisi gibi düşünebilirsin. Görev, serinin son 30 değerine bakıp bir sonraki değeri tahmin etmek. Bunun için seri üzerinde 30 adımlık bir pencere kaydırılır: Her pencere bir eğitim örneği, pencereden hemen sonraki değer de onun hedefidir. 1.000 adımlık seriden 970 pencere çıkar.

*Şekil 20.3: Üstte serinin tamamı: İlk 776 pencere eğitim, son 194 pencere test için ayrılır. Altta tek bir eğitim örneği: Mavi 30 nokta girdi, turuncu nokta hedeftir. (Şekil PDF'te.)*

> **DİKKAT · Zaman serisini rastgele bölme**
>
> Bölüm 6'daki rastgele bölme, zaman serilerinde sızıntıya yol açar: Model, tahmin etmesi istenen dönemin ortasından ve sonrasından örnekler görmüş olur. Gerçek hayatta yalnızca geçmiş bilinir; bu yüzden seriyi zamana göre böl: geçmişle eğit, gelecekte test et. Eğitim kümesindeki pencereleri kendi aralarında karıştırmak ise sorun değildir, çünkü her pencere kendi geçmişini zaten içinde taşır.

### 1–3) Veri ve basit karşılaştırma modelleri

In [ ]:
"""Örnek 2: LSTM ile zaman serisi tahmini (PyTorch).

Veri kod içinde üretilir; internetten indirme gerekmez.
Kurulum:  pip install torch scikit-learn
"""
import numpy as np
import torch
from torch import nn
from sklearn.linear_model import LinearRegression

torch.manual_seed(0)

In [ ]:
# 1) Veri: gürültülü bir sinüs dalgası (periyot 50 adım, gürültünün std'si 0,1)
rng = np.random.default_rng(0)
t = np.arange(1000)
seri = np.sin(2 * np.pi * t / 50) + 0.1 * rng.standard_normal(1000)

In [ ]:
# 2) Kayan pencere: son 30 değere bakıp bir sonrakini tahmin et
P = 30
X = np.stack([seri[i:i + P] for i in range(len(seri) - P)])[..., None]  # (970, 30, 1)
y = seri[P:, None]                                                      # (970, 1)
n = int(0.8 * len(X))  # zaman sırasını koru: geçmişle eğit, gelecekte test et
X_egt, y_egt, X_test, y_test = X[:n], y[:n], X[n:], y[n:]

In [ ]:
# 3) Önce basit modeller: her zaman bir başlangıç modeliyle karşılaştır
def rmse(tahmin):
    return np.sqrt(np.mean((tahmin - y_test) ** 2))


print(f"Naif model ('sonraki = son değer') RMSE: {rmse(X_test[:, -1]):.3f}")
dogrusal = LinearRegression().fit(X_egt[..., 0], y_egt)
print(f"Doğrusal regresyon RMSE: {rmse(dogrusal.predict(X_test[..., 0])):.3f}")

Derin modele geçmeden önce iki basit modelin hatası ölçülür (Bölüm 19.1, 4. adım). Naif model “bir sonraki değer, son değerle aynı olur” der. Doğrusal regresyon (Bölüm 3) ise son 30 değerin ağırlıklı toplamını öğrenir. Bu iki sonuç rastgelelikten etkilenmez; kendi bilgisayarında da aynı çıkmalı:

**Beklenen çıktı** · Çıktının ilk iki satırı (kesin)

```text
Naif model ('sonraki = son değer') RMSE: 0.149
Doğrusal regresyon RMSE: 0.095
```

### 4) LSTM modeli

In [ ]:
# 4) LSTM modeli
class LSTMTahminci(nn.Module):
    def __init__(self, gizli=32):
        super().__init__()
        self.rnn = nn.LSTM(input_size=1, hidden_size=gizli, batch_first=True)
        self.cikis = nn.Linear(gizli, 1)

    def forward(self, x):                     # x: (yığın, 30, 1)
        tum_h, (h_son, c_son) = self.rnn(x)  # h_son: (katman sayısı, yığın, 32)
        return self.cikis(h_son[-1])          # son adımın gizli durumu → tahmin


model = LSTMTahminci()
print("Parametre sayısı:", sum(p.numel() for p in model.parameters()))

X_egt_t = torch.tensor(X_egt, dtype=torch.float32)
y_egt_t = torch.tensor(y_egt, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

kayip_fn = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
yukleyici = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_egt_t, y_egt_t), batch_size=32, shuffle=True)

`nn.LSTM`'in üç ayarı vardır: `input_size=1` (her adımda tek bir sayı girer), `hidden_size=32` (gizli durum ve hücre durumu 32 boyutludur) ve `batch_first=True` (girdinin şekli yığın, adım, özellik sırasındadır). LSTM iki şey döndürür: bütün adımların gizli durumları (`tum_h`) ve son adımın `(h_son, c_son)` çifti. `h_son`'un şekli (katman sayısı, yığın, 32) olduğu için `h_son[-1]` son katmanın son adımdaki gizli durumunu, yani dizinin bir tür özetini verir. Doğrusal katman bu özetten tek bir tahmin üretir; bu, Bölüm 16.7'deki “çoka bir” yapısıdır.

Parametre sayısı 4.513'tür. LSTM katmanında 4 · 32 · (1 + 32 + 2) = 4.480, çıkış katmanında 32 + 1 = 33 parametre vardır. Buradaki 4, Bölüm 16.5'teki dört denklemden (f, i, g, o) gelir. Her biri için 32 nöronun girdiye bağlanan ağırlıkları (1), önceki gizli duruma bağlanan ağırlıkları (32) ve PyTorch'un tuttuğu iki sapma vektörü (2) bulunur.

### 5) Eğitim ve 6) Test

In [ ]:
# 5) Eğitim döngüsü: CNN örneğindeki dört adımın aynısı
for epok in range(1, 31):
    model.train()
    toplam = 0.0
    for xb, yb in yukleyici:
        opt.zero_grad()
        kayip = kayip_fn(model(xb), yb)
        kayip.backward()
        opt.step()
        toplam += kayip.item() * len(xb)
    if epok % 10 == 0:
        print(f"Epok {epok} | eğitim MSE {toplam / n:.4f}")

In [ ]:
# 6) Test
model.eval()
with torch.no_grad():
    tahmin = model(X_test_t).numpy()
print(f"LSTM RMSE: {rmse(tahmin):.3f}")

Eğitim döngüsü CNN örneğindekiyle aynıdır; yalnızca kayıp fonksiyonu, regresyon olduğu için ortalama kare hatadır (`nn.MSELoss`, Bölüm 3.2). Testte tahminler NumPy dizisine çevrilir ve basit modellerle aynı `rmse` fonksiyonuyla ölçülür.

**Beklenen çıktı** (PyTorch 2.14, işlemci, tohum 0)

```text
Naif model ('sonraki = son değer') RMSE: 0.149
Doğrusal regresyon RMSE: 0.095
Parametre sayısı: 4513
Epok 10 | eğitim MSE 0.0144
Epok 20 | eğitim MSE 0.0147
Epok 30 | eğitim MSE 0.0135
LSTM RMSE: 0.098
```

> **NE GÖRMELİSİN? · Örnek 2**
>
> Aynı kod 10 farklı tohumla (0–9) çalıştırıldığında sonuçlar şu aralıklarda kaldı:
> - **Parametre sayısı:** 4513 (kesin). **Naif RMSE:** 0.149, **doğrusal regresyon RMSE:** 0.095 (kesin).
> - **Eğitim MSE'si:** 10. epokta 0,013–0,016, 30. epokta 0,012–0,014. Gürültünün varyansı 0,1² = 0,01 olduğundan bunun çok altına inmesi beklenmez.
> - **LSTM RMSE:** yaklaşık 0,10 (10 denemede 0,093 ile 0,105 arası).

*Şekil 20.4: Test döneminin ilk 100 adımı. Naif tahmin gerçek değerleri bir adım geriden ve gürültüsüyle birlikte izliyor; LSTM ise gürültüyü büyük ölçüde süzüp düzgün sinüs dalgasını takip ediyor. LSTM tahminleri yukarıdaki çalıştırmadan (tohum 0) alınmıştır. (Şekil PDF'te.)*

### Sonucu yorumlamak

**Tablo 20.3: Örnek 2'de modellerin karşılaştırılması**

| Model | Test RMSE | Açıklama |
|---|---|---|
| Naif (“sonraki = son değer”) | 0,149 | Kesin |
| LSTM | 0,098 | Tohum 0; 10 denemede 0,093–0,105 |
| Doğrusal regresyon (son 30 değer) | 0,095 | Kesin |
| Gerçek sinüsü bilen kusursuz model | ≈ 0,089 | Test dönemindeki gürültünün kendisi: ulaşılabilecek en iyi değer |

LSTM naif modeli açıkça geçiyor ama doğrusal regresyonu genellikle geçemiyor: 10 denemenin yalnızca birinde (0,093) ondan biraz daha iyi sonuç verdi. Bu bir hata değil. Gürültüsüz bir sinüs dalgasında her değer, önceki iki değerin doğrusal bir birleşimi olarak tam yazılabilir; yani bu seri, doğrusal bir model için ideal bir problemdir. Üstelik iki model de gürültünün koyduğu sınıra yaklaşmıştır: Gerçek sinüsü bilen kusursuz bir model bile test döneminde yaklaşık 0,089 hata yapardı. Bu, Bölüm 7'deki indirgenemez hatadır.

Bu küçük deney, araştırmalarda da görülen bir durumu yansıtıyor. Nicosia Üniversitesi ve Atina Ulusal Teknik Üniversitesi'nden Makridakis ve arkadaşları, M3 tahmin yarışmasının 1.045 aylık serisinde, aralarında LSTM'nin de bulunduğu makine öğrenmesi yöntemlerinin basit istatistiksel yöntemlerin gerisinde kaldığını ve çok daha fazla hesaplama gerektirdiğini göstermiştir [79]. Monash Üniversitesi'nden Hewamalage ve arkadaşları ise RNN'lerin doğru ön işlemeyle birçok durumda güçlü bir seçenek olduğunu, ama her derde deva olmadığını vurgular [80]. Çıkarılacak ders: Derin bir modeli her zaman basit modellerle karşılaştır. LSTM gibi modellerin avantajı genellikle ilişkilerin doğrusal olmadığı, uzun bağımlılıkların ve çok sayıda benzer serinin bulunduğu problemlerde ortaya çıkar.

### Kendin dene

**4.** LSTM yerine GRU kullan. Aşağıdaki iki satırı değiştirmen yeterli. GRU'da dört yerine üç denklem takımı olduğu için parametre sayısı 3 · 32 · 35 + 33 = 3.393'e iner.

*GRU için yapılacak değişiklik* (yalnızca okumak için)

```python
self.rnn = nn.GRU(input_size=1, hidden_size=gizli, batch_first=True)  # __init__ içinde
tum_h, h_son = self.rnn(x)  # forward içinde: GRU'da ayrı bir hücre durumu (c) yoktur
```

**5.** Gürültünün standart sapmasını 0,1'den 0,3'e çıkar. Üç modelin RMSE'si nasıl değişiyor?